# 02. 메인 그리드: 후보 모델 × 묶음 조건 (Stellantis N.V., 차체 단위 CV)
설계서 6절의 후보를 같은 조건에서 돌리고, **통과 기준은 결과를 보기 전에 아래와 같이 고정**한다(셀 실행 후 수정하지 않는다).

| # | 기준 | 판정 |
|---|---|---|
| 1 | 1단계: 묶음 개수 오차(MAE)가 고정 불량률 기준보다 작다 | 시드 평균 비교 |
| 2 | 2단계: 예산 10% 포착률의 95% 구간 하한 > 0.10 (무작위 기대값) | 차체 부트스트랩 |
| 3 | 같은 위치 안: 위치별 예산 10% 포착률 하한 > 0.10 **그리고** 같은 위치 안 AUROC 하한 > 0.5 | 차체 부트스트랩 |
| 4 | KAMP형 조건(`k8_1`, `k8_05`) 둘 다에서 1~3 유지 | — |

판정 대상은 우리 모델(A·B·C·D)이다. 기준·상한 모델은 비교선으로만 쓴다.
`loc_rate`는 학습 폴드에서 차체를 뺀 위치별 불량률이라 같은 위치 안 점수가 leave-one-out 인공물(AUROC < 0.5)이 되므로 위치 안 지표를 "해당 없음"으로 둔다.

In [1]:
import sys, time, numpy as np, pandas as pd, torch
sys.path.insert(0, ".")
from joblib import Parallel, delayed
from wl import data, runner, bags
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 50)

MODELS = ["const_rate", "loc_rate", "iforest", "autoencoder",
          "A_dllp_mlp", "B_count_mlp", "C_prior_two", "D_pseudo_logreg",
          "UB_logreg", "UB_catboost"]
OURS = ["A_dllp_mlp", "B_count_mlp", "C_prior_two", "D_pseudo_logreg"]
CONDS = list(bags.CONDITIONS)
SEEDS = list(range(5))
N_GPU = torch.cuda.device_count()
RERUN = False

## 1. 학습·OOF 예측 (GPU 2장에 작업을 나눠 병렬 실행)

In [2]:
def _task(i, m, c, s):
    return runner.run_one(m, c, s, gpu=i % max(N_GPU, 1))

name = "main_grid"
if (runner.RES / f"{name}.pkl").exists() and not RERUN:
    runs = runner.load_runs(name)
else:
    tasks = [(m, c, s) for m in MODELS for c in CONDS for s in SEEDS]
    t = time.time()
    runs = Parallel(n_jobs=8, verbose=5)(delayed(_task)(i, *k) for i, k in enumerate(tasks))
    print(f"{len(runs)}개 실행, {time.time()-t:.0f}초")
    runner.save_runs(runs, name)
by = {}
for r in runs: by.setdefault((r["model"], r["cond"]), []).append(r)

## 2. 지표 + 차체 단위 부트스트랩 95% 구간

In [3]:
def _boot(key, n_boot=200):
    return key, runner.bootstrap(by[key], n_boot=n_boot)
t = time.time()
boots = dict(Parallel(n_jobs=16)(delayed(_boot)(k) for k in by))
print(f"부트스트랩 {time.time()-t:.0f}초")
WL = ["wl_auroc", "wl_recall@0.05", "wl_recall@0.1", "wl_recall@0.2"]
for (m, c), b in boots.items():
    if m == "loc_rate": b.loc[WL, :] = np.nan

def fmt(m, c, k, d=3):
    r = boots[(m, c)].loc[k]
    return "n/a" if np.isnan(r["mean"]) else f"{r['mean']:.{d}f} [{r['lo']:.{d}f}, {r['hi']:.{d}f}]"

def table(c, keys):
    return pd.DataFrame({k: {m: fmt(m, c, k) for m in MODELS} for k in keys})

부트스트랩 15초


### 2-1. 원본 조건 (`orig`: 20묶음, 불량률 4%)

In [4]:
KEYS = ["mae", "auroc", "wl_auroc", "recall@0.1", "wl_recall@0.1", "recall@5x"]
table("orig", KEYS)

,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"2.328 [1.979, 4.801]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.196 [0.192, 0.202]"
loc_rate,"2.232 [1.829, 4.482]","0.804 [0.764, 0.855]",n/a,"0.491 [0.441, 0.568]",n/a,"0.705 [0.657, 0.761]"
iforest,"2.328 [1.979, 4.801]","0.592 [0.553, 0.636]","0.447 [0.359, 0.525]","0.133 [0.067, 0.181]","0.044 [0.028, 0.094]","0.255 [0.170, 0.327]"
autoencoder,"2.328 [1.979, 4.801]","0.583 [0.532, 0.625]","0.572 [0.506, 0.637]","0.115 [0.060, 0.166]","0.093 [0.069, 0.144]","0.272 [0.184, 0.321]"
A_dllp_mlp,"2.160 [1.858, 4.596]","0.713 [0.627, 0.770]","0.550 [0.470, 0.618]","0.294 [0.184, 0.389]","0.141 [0.092, 0.179]","0.469 [0.296, 0.578]"
B_count_mlp,"2.694 [2.197, 5.830]","0.673 [0.610, 0.714]","0.491 [0.377, 0.566]","0.223 [0.131, 0.297]","0.114 [0.069, 0.149]","0.385 [0.236, 0.501]"
C_prior_two,"2.232 [1.879, 4.688]","0.625 [0.533, 0.684]","0.444 [0.342, 0.527]","0.167 [0.084, 0.232]","0.082 [0.051, 0.110]","0.337 [0.210, 0.448]"
D_pseudo_logreg,"2.210 [1.823, 4.574]","0.702 [0.636, 0.745]","0.426 [0.322, 0.510]","0.187 [0.111, 0.238]","0.070 [0.040, 0.109]","0.422 [0.261, 0.517]"
UB_logreg,"2.276 [1.946, 4.581]","0.740 [0.704, 0.779]","0.346 [0.260, 0.424]","0.309 [0.212, 0.371]","0.062 [0.027, 0.094]","0.466 [0.419, 0.536]"
UB_catboost,"1.973 [1.514, 4.064]","0.824 [0.780, 0.872]","0.531 [0.428, 0.631]","0.542 [0.466, 0.635]","0.097 [0.057, 0.167]","0.708 [0.624, 0.798]"


### 2-2. 묶음만 8개로 줄인 조건 (`k8_4`)

In [5]:
table("k8_4", KEYS)

,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"4.395 [3.023, 7.842]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.197 [0.193, 0.202]"
loc_rate,"4.715 [3.249, 8.027]","0.804 [0.764, 0.855]",n/a,"0.491 [0.441, 0.568]",n/a,"0.689 [0.641, 0.752]"
iforest,"4.395 [3.023, 7.842]","0.592 [0.553, 0.636]","0.447 [0.359, 0.525]","0.133 [0.067, 0.181]","0.044 [0.028, 0.094]","0.261 [0.183, 0.328]"
autoencoder,"4.395 [3.023, 7.842]","0.583 [0.532, 0.625]","0.572 [0.506, 0.637]","0.115 [0.060, 0.166]","0.093 [0.069, 0.144]","0.277 [0.182, 0.325]"
A_dllp_mlp,"3.840 [2.625, 7.376]","0.599 [0.521, 0.656]","0.514 [0.457, 0.555]","0.144 [0.061, 0.212]","0.094 [0.066, 0.119]","0.273 [0.146, 0.414]"
B_count_mlp,"4.790 [4.010, 9.269]","0.559 [0.509, 0.594]","0.482 [0.415, 0.541]","0.092 [0.044, 0.135]","0.076 [0.055, 0.109]","0.228 [0.142, 0.343]"
C_prior_two,"4.029 [2.617, 7.416]","0.530 [0.499, 0.568]","0.497 [0.444, 0.546]","0.084 [0.047, 0.127]","0.074 [0.057, 0.106]","0.183 [0.123, 0.256]"
D_pseudo_logreg,"4.337 [3.156, 7.582]","0.713 [0.660, 0.750]","0.434 [0.334, 0.519]","0.200 [0.125, 0.247]","0.068 [0.034, 0.103]","0.441 [0.275, 0.536]"
UB_logreg,"4.635 [2.983, 7.804]","0.740 [0.704, 0.779]","0.346 [0.260, 0.424]","0.309 [0.212, 0.371]","0.062 [0.027, 0.094]","0.465 [0.409, 0.534]"
UB_catboost,"3.743 [2.509, 7.103]","0.824 [0.780, 0.872]","0.531 [0.428, 0.631]","0.542 [0.466, 0.635]","0.097 [0.057, 0.167]","0.691 [0.613, 0.776]"


### 2-3. KAMP형: 8묶음·불량률 1% (`k8_1`)

In [6]:
table("k8_1", KEYS)

,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"1.442 [1.201, 2.488]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.049 [0.047, 0.050]"
loc_rate,"1.481 [1.238, 2.551]","0.717 [0.678, 0.788]",n/a,"0.537 [0.441, 0.617]",n/a,"0.450 [0.356, 0.534]"
iforest,"1.442 [1.201, 2.488]","0.610 [0.550, 0.662]","0.431 [0.308, 0.550]","0.137 [0.054, 0.232]","0.032 [0.009, 0.113]","0.063 [0.012, 0.144]"
autoencoder,"1.442 [1.201, 2.488]","0.629 [0.565, 0.683]","0.632 [0.570, 0.698]","0.154 [0.082, 0.238]","0.145 [0.084, 0.203]","0.084 [0.021, 0.138]"
A_dllp_mlp,"1.425 [1.174, 2.485]","0.611 [0.561, 0.647]","0.482 [0.425, 0.541]","0.094 [0.048, 0.129]","0.069 [0.037, 0.108]","0.055 [0.012, 0.108]"
B_count_mlp,"1.703 [1.516, 3.167]","0.559 [0.498, 0.617]","0.531 [0.419, 0.627]","0.137 [0.080, 0.217]","0.099 [0.051, 0.158]","0.032 [0.000, 0.099]"
C_prior_two,"1.415 [1.276, 2.656]","0.525 [0.473, 0.576]","0.396 [0.334, 0.461]","0.063 [0.011, 0.130]","0.056 [0.032, 0.098]","0.050 [0.000, 0.089]"
D_pseudo_logreg,"1.394 [1.172, 2.495]","0.658 [0.597, 0.706]","0.295 [0.220, 0.364]","0.168 [0.087, 0.226]","0.061 [0.016, 0.092]","0.084 [0.049, 0.166]"
UB_logreg,"1.384 [1.227, 2.521]","0.727 [0.673, 0.774]","0.222 [0.153, 0.284]","0.295 [0.200, 0.362]","0.021 [0.000, 0.044]","0.178 [0.112, 0.283]"
UB_catboost,"1.417 [1.049, 2.566]","0.794 [0.723, 0.854]","0.440 [0.334, 0.554]","0.558 [0.442, 0.665]","0.075 [0.046, 0.128]","0.347 [0.264, 0.435]"


### 2-4. KAMP형: 8묶음·불량률 0.5% (`k8_05`)

In [7]:
table("k8_05", KEYS)

,mae,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"0.903 [0.860, 1.591]","0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.024 [0.023, 0.026]"
loc_rate,"0.924 [0.858, 1.628]","0.593 [0.440, 0.738]",n/a,"0.407 [0.242, 0.562]",n/a,"0.317 [0.182, 0.461]"
iforest,"0.903 [0.860, 1.591]","0.614 [0.553, 0.677]","0.443 [0.327, 0.532]","0.160 [0.036, 0.258]","0.028 [0.002, 0.102]","0.020 [0.000, 0.070]"
autoencoder,"0.903 [0.860, 1.591]","0.546 [0.468, 0.623]","0.508 [0.391, 0.637]","0.120 [0.047, 0.187]","0.088 [0.047, 0.169]","0.035 [0.000, 0.107]"
A_dllp_mlp,"0.901 [0.891, 1.637]","0.598 [0.512, 0.694]","0.461 [0.359, 0.572]","0.156 [0.067, 0.248]","0.068 [0.025, 0.134]","0.009 [0.003, 0.024]"
B_count_mlp,"1.169 [1.000, 2.052]","0.487 [0.405, 0.583]","0.506 [0.390, 0.609]","0.080 [0.015, 0.207]","0.118 [0.038, 0.201]","0.020 [0.000, 0.090]"
C_prior_two,"0.996 [0.940, 1.698]","0.562 [0.491, 0.642]","0.465 [0.348, 0.573]","0.160 [0.088, 0.254]","0.062 [0.029, 0.154]","0.038 [0.000, 0.101]"
D_pseudo_logreg,"0.888 [0.857, 1.584]","0.552 [0.478, 0.647]","0.358 [0.242, 0.452]","0.100 [0.030, 0.170]","0.038 [0.002, 0.072]","0.040 [0.000, 0.099]"
UB_logreg,"0.866 [0.843, 1.584]","0.659 [0.601, 0.724]","0.230 [0.163, 0.305]","0.240 [0.138, 0.338]","0.000 [0.000, 0.035]","0.080 [0.022, 0.180]"
UB_catboost,"0.994 [0.834, 1.673]","0.720 [0.624, 0.805]","0.343 [0.229, 0.442]","0.360 [0.197, 0.469]","0.014 [0.007, 0.071]","0.100 [0.040, 0.204]"


## 3. 통과 기준 판정 (위 표의 기준을 그대로 적용)

In [8]:
def verdict(m, c):
    mae_ok = boots[(m, c)].loc["mae", "mean"] < boots[("const_rate", c)].loc["mae", "mean"]
    r10 = boots[(m, c)].loc["recall@0.1", "lo"] > 0.10
    wl = boots[(m, c)].loc["wl_recall@0.1", "lo"] > 0.10 and boots[(m, c)].loc["wl_auroc", "lo"] > 0.5
    return mae_ok, r10, wl

rows = []
for m in OURS:
    row = {"모델": m}
    for c in CONDS:
        a, b, w = verdict(m, c)
        row[c] = f"{'O' if a else 'X'}{'O' if b else 'X'}{'O' if w else 'X'}"
    kl = [verdict(m, c) for c in ("k8_1", "k8_05")]
    row["기준4(KAMP형 유지)"] = "통과" if all(all(v) for v in kl) else "미통과"
    row["전 조건 통과"] = "통과" if all(all(verdict(m, c)) for c in CONDS) else "미통과"
    rows.append(row)
print("표기: [기준1 MAE][기준2 포착률][기준3 위치안] 순서, O=충족 X=미충족")
pd.DataFrame(rows)

표기: [기준1 MAE][기준2 포착률][기준3 위치안] 순서, O=충족 X=미충족


,모델,orig,k8_4,k8_1,k8_05,기준4(KAMP형 유지),전 조건 통과
0,A_dllp_mlp,OOX,OXX,OXX,OXX,미통과,미통과
1,B_count_mlp,XOX,XXX,XXX,XXX,미통과,미통과
2,C_prior_two,OXX,OXX,OXX,XXX,미통과,미통과
3,D_pseudo_logreg,OOX,OOX,OXX,OXX,미통과,미통과
